# Few-Shot PCB Defect Segmentation

Presentation notebook for the frozen evidence revision. Core implementation remains in `src/` and `scripts/`; this notebook only verifies and reads compact public evidence.

## 1. Research Question and Disclosure

Can anomaly-guided SAM2 prompting improve few-shot PCB defect segmentation over calibrated DINOv2 anomaly masks, especially for small or thin defects?

Paper-facing claims use the frozen VisA PCB matrices only. Earlier development inspected VisA test results, so the official test set is not an untouched holdout. Oracle/test-optimal thresholds remain diagnostic and are never mixed with calibrated primary-mask comparisons.

In [ ]:
from __future__ import annotations

import csv
import hashlib
import json
from pathlib import Path


def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "docs/evidence").is_dir():
            return candidate
    raise FileNotFoundError("run the notebook from inside the repository")


PROJECT_ROOT = find_project_root(Path.cwd())
EVIDENCE_RELATIVE = Path("docs/evidence/generated")
EVIDENCE_DIR = PROJECT_ROOT / EVIDENCE_RELATIVE


def require_file(path: Path) -> Path:
    if not path.is_file():
        raise FileNotFoundError(f"missing frozen evidence: {path}")
    return path


def read_csv(name: str) -> list[dict[str, str]]:
    with require_file(EVIDENCE_DIR / name).open(newline="", encoding="utf-8") as handle:
        return list(csv.DictReader(handle))


def read_json(name: str) -> dict:
    return json.loads(require_file(EVIDENCE_DIR / name).read_text(encoding="utf-8"))


def sha256(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()

## 2. Frozen Protocol and Evidence Integrity

VisA categories `pcb1`–`pcb4` are evaluated on fold 0 with `k in {1, 2, 4}` normal supports and seeds `4880`–`4884`. The seeds repeat support sampling on the same official test images; they are not independent folds. DeepPCB remains secondary because its boxes are not segmentation masks.

No model parameters are trained or fine-tuned. Each run builds a support-feature memory bank, optionally applies a PatchCore-style coreset, calibrates on normal validation images, and performs frozen-model inference.

In [ ]:
manifest = read_json("completion_manifest.json")
assert manifest["ready_for_writing"] is True
assert manifest["readiness_scope"] == "complete_paper_evidence"
assert len(manifest["source_runs"]) == 412
for item in manifest["generated_files"]:
    path = require_file(EVIDENCE_DIR / item["path"])
    assert sha256(path) == item["sha256"], path
validation = read_json("validation_summary.json")
assert validation["primary"]["ok"] and validation["primary"]["bad"] == 0
assert validation["ablations"]["ok"] and validation["ablations"]["bad"] == 0
assert validation["method_invariants"]["ok"]
print("ready_for_writing:", manifest["ready_for_writing"])
print("source runs:", len(manifest["source_runs"]))
print("primary matrix:", validation["primary"]["complete"], "/", validation["primary"]["expected"])
print("ablation matrix:", validation["ablations"]["complete"], "/", validation["ablations"]["expected"])
print("method invariants:", validation["method_invariants"])

## 3. Method Summary

A few normal supports populate a frozen DINOv2 memory bank. Global and local patch scores form a multi-scale anomaly map. Its normal-only calibrated proposal supplies SAM2 point/box prompts. The final anomaly-consistent mask is the exact intersection of the calibrated proposal and guided SAM2 mask.

![Method diagram](../docs/evidence/generated/method_figure.png)

## 4. Metrics and Oracle Boundary

Primary heatmap masks use the normal-validation `q=0.995` threshold. SAM2 methods use saved binary masks on identical test images. Oracle thresholds are test-optimal diagnostics only. Pixel AUROC uses a deterministic per-run pixel sample; calibrated F1/IoU and AUPRO use full-resolution maps as documented below.

In [ ]:
metric_definitions = read_json("metric_definitions.json")
oracle_rows = read_csv("oracle_diagnostics.csv")
print("oracle diagnostic rows:", len(oracle_rows))
for name, definition in metric_definitions.items():
    if name != "schema_version":
        print(f"{name}: {definition}")

## 5. Primary Category-Macro Mask F1

The compact table below is loaded from the checksum-covered primary summary. Intervals are support-seed intervals, not independent-test-set intervals. `sam2_only` is a deterministic `k=0`, seed-0 descriptive baseline.

In [ ]:
primary_summary = read_csv("primary_summary.csv")
macro_f1 = [
    row for row in primary_summary
    if row["category"] == "macro" and row["metric"] == "mean_anomaly_mask_f1"
]
for row in macro_f1:
    interval = "descriptive" if not row["ci_low"] else f"[{float(row['ci_low']):.4f}, {float(row['ci_high']):.4f}]"
    print(f"{row['method']:30s} k={row['k']:>1s} mean={float(row['mean']):.4f} interval={interval}")

## 6. Paired Evidence and Supporting Comparisons

The planned primary delta is candidate minus baseline on anomalous images, with category and shot count fixed and shared support-seed/test-image resampling. Supporting comparisons use identical pairings.

In [ ]:
conclusion = read_json("evidence_conclusion.json")
assert conclusion["outcome"] == "anomaly_consistent_sam2_improves_robustly"
print("outcome:", conclusion["outcome"])
for metric in ("f1", "iou"):
    result = conclusion[metric]
    print(f"{metric.upper()} delta={result['mean_delta']:.4f}, 95% CI [{result['ci_low']:.4f}, {result['ci_high']:.4f}]")
supporting = read_json("baseline_paired_statistics.json")["comparisons"]
for name, comparison in supporting.items():
    f1 = comparison["overall"]["f1"]
    print(f"{name}: F1 delta={f1['mean_delta']:.4f}, 95% CI [{f1['ci_low']:.4f}, {f1['ci_high']:.4f}]")

## 7. Curated Success and Failure Cases

Each category contributes one positive and one negative guided-SAM2 F1 delta. These are deterministic examples, not aggregate evidence.

| Category | Success | Failure |
| --- | --- | --- |
| PCB1 | ![PCB1 success](../docs/evidence/generated/qualitative_figures/pcb1_success.png) | ![PCB1 failure](../docs/evidence/generated/qualitative_figures/pcb1_failure.png) |
| PCB2 | ![PCB2 success](../docs/evidence/generated/qualitative_figures/pcb2_success.png) | ![PCB2 failure](../docs/evidence/generated/qualitative_figures/pcb2_failure.png) |
| PCB3 | ![PCB3 success](../docs/evidence/generated/qualitative_figures/pcb3_success.png) | ![PCB3 failure](../docs/evidence/generated/qualitative_figures/pcb3_failure.png) |
| PCB4 | ![PCB4 success](../docs/evidence/generated/qualitative_figures/pcb4_success.png) | ![PCB4 failure](../docs/evidence/generated/qualitative_figures/pcb4_failure.png) |

In [ ]:
qualitative = read_csv("qualitative_manifest.csv")
assert len(qualitative) == 8
for row in qualitative:
    delta = float(row["sam2_delta_f1"])
    assert (row["role"] == "success" and delta > 0) or (row["role"] == "failure" and delta < 0)
    print(row["category"], row["role"], row["sample_id"], f"delta={delta:+.4f}")

## 8. Evidence-Led Conclusion and Limitations

The frozen conclusion is **anomaly_consistent_sam2_improves_robustly** relative to calibrated multi-scale DINOv2: F1 `+0.0666` (95% CI `[0.0538, 0.0788]`) and IoU `+0.0565` (`[0.0462, 0.0665]`). The benefit is robust for every category and shot count, and is largest for thin defects. The large-area stratum is inconclusive.

Supporting evidence must stay nuanced: multi-scale versus single-scale DINOv2 is inconclusive; multi-scale is robustly better than this repository's PatchCore-style baseline; guided multi-scale versus guided single-scale SAM2 is inconclusive. Ablations use only `k=4`, seed `4880` and are descriptive. The study is fold-0 only, reuses official test images across support seeds, has prior test-set exposure, and makes no state-of-the-art or broad first-method claim.

## 9. Reproduction and Writing Handoff

Run the tested pipeline and matrix commands in `README.md` and `docs/autodl_data_setup.md`. The exact frozen audit and finalizer invocations are recorded in `frozen_audit_manifest.json` and `completion_manifest.json`. Use `docs/evidence/generated/evidence_index.md` to map every planned claim/table/figure to its source artifact, and `docs/citation_inventory.md` for attribution and novelty boundaries. Manuscript drafting remains a separate human step.